# 04. Расчет и визуализация DB-weighted мутационного спектра

Основная идея: частые в популяционных базах замещения должны давать больший вклад в спектр, а ненаблюдавшиеся замещения получают минимальный вклад через псевдокаунт.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

INPUT_PATH = Path(
    "../results/mutational_spectrum_groups/"
    "spectrum_groups_from_one_class_T95_with_codon_phyloP100way_"
    "with_db_spectrum_weights_v1.tsv"
)

OUTPUT_DIR = Path("../results/mutational_spectrum_results")
PLOTS_DIR = OUTPUT_DIR / "plots_db_weighted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SPECTRUM_OUTPUT = OUTPUT_DIR / "mutation_spectrum_db_weighted_frequencies_T95_phyloP100way_v1.tsv"

WEIGHT_COL = "combined_db_spectrum_weight"
SUBSTITUTION_COL = "substitution_type_12"

SUBSTITUTION_ORDER = [
    "A>C", "A>G", "A>T",
    "C>A", "C>G", "C>T",
    "G>A", "G>C", "G>T",
    "T>A", "T>C", "T>G",
]

GROUP_COLS = [
    "spectrum_group_primary_T95",
    "spectrum_group_strict_T95",
    "spectrum_group_posthoc",
]


## Как считается вес замещения

В `03a` для каждого возможного замещения был рассчитан вес:

```text
combined_db_spectrum_weight_i = max(gnomAD_count_i + Helix_count_i, 1) /
                                ((gnomAD_total + Helix_total) × ref_base_count_i)
```

In [3]:
df = pd.read_csv(INPUT_PATH, sep="	", low_memory=False)

df = df[df[SUBSTITUTION_COL].isin(SUBSTITUTION_ORDER)].copy()
df[WEIGHT_COL] = pd.to_numeric(df[WEIGHT_COL], errors="coerce")

df[["variant_id", SUBSTITUTION_COL, WEIGHT_COL] + GROUP_COLS].head()

,variant_id,substitution_type_12,combined_db_spectrum_weight,spectrum_group_primary_T95,spectrum_group_strict_T95,spectrum_group_posthoc
0,m.1G>T,G>T,1.826509e-09,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
1,m.1G>A,G>A,1.826509e-09,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
2,m.1G>C,G>C,1.826509e-09,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
3,m.2A>T,A>T,7.731652e-10,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
4,m.2A>C,A>C,7.731652e-10,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95


## Counting MutSpec

for each group `g` and ech type of sbs `t` sum al weights for all variants of the type:

```text
weighted_sum(g, t) = Σ combined_db_spectrum_weight_i
```

Then norm inside the group:

```text
weighted_frequency(g, t) = weighted_sum(g, t) / Σ weighted_sum(g, all substitution types)
```

 `weighted_frequency` - final column

In [ ]:
def compute_weighted_spectrum(data, group_col):
    tmp = data[data[group_col] != "exclude"].copy()

    spectrum = (
        tmp
        .groupby([group_col, SUBSTITUTION_COL], dropna=False)
        .agg(
            n_variants=("variant_id", "count"),
            weighted_sum=(WEIGHT_COL, "sum"),
        )
        .reset_index()
        .rename(columns={group_col: "group_name", SUBSTITUTION_COL: "substitution_type"})
    )

    group_names = list(tmp[group_col].dropna().unique())
    full_index = pd.MultiIndex.from_product(
        [group_names, SUBSTITUTION_ORDER],
        names=["group_name", "substitution_type"],
    )

    spectrum = (
        spectrum
        .set_index(["group_name", "substitution_type"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    spectrum["group_col"] = group_col
    spectrum["total_weight"] = spectrum.groupby("group_name")["weighted_sum"].transform("sum")
    spectrum["weighted_frequency"] = spectrum["weighted_sum"] / spectrum["total_weight"]

    return spectrum[
        [
            "group_col",
            "group_name",
            "substitution_type",
            "n_variants",
            "weighted_sum",
            "total_weight",
            "weighted_frequency",
        ]
    ]


spectrum_df = pd.concat(
    [compute_weighted_spectrum(df, group_col) for group_col in GROUP_COLS],
    ignore_index=True,
)

spectrum_df.to_csv(SPECTRUM_OUTPUT, sep="	", index=False)

spectrum_df.head(20)

## Plotting

In [ ]:
def plot_single_spectrum(spectrum_df, group_col, group_name):
    plot_df = (
        spectrum_df[
            (spectrum_df["group_col"] == group_col)
            & (spectrum_df["group_name"] == group_name)
        ]
        .set_index("substitution_type")
        .reindex(SUBSTITUTION_ORDER)
        .reset_index()
    )

    x = np.arange(len(SUBSTITUTION_ORDER))

    plt.figure(figsize=(10, 5))
    plt.bar(x, plot_df["weighted_frequency"])
    plt.xticks(x, SUBSTITUTION_ORDER, rotation=45, ha="right")
    plt.xlabel("Substitution type")
    plt.ylabel("DB-weighted frequency")
    plt.title(f"{group_name}{group_col}")
    plt.tight_layout()

    output_path = PLOTS_DIR / f"single_{group_col}_{group_name}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_two_group_spectrum(spectrum_df, group_col, group_1, group_2):
    df1 = (
        spectrum_df[
            (spectrum_df["group_col"] == group_col)
            & (spectrum_df["group_name"] == group_1)
        ]
        .set_index("substitution_type")
        .reindex(SUBSTITUTION_ORDER)
        .reset_index()
    )

    df2 = (
        spectrum_df[
            (spectrum_df["group_col"] == group_col)
            & (spectrum_df["group_name"] == group_2)
        ]
        .set_index("substitution_type")
        .reindex(SUBSTITUTION_ORDER)
        .reset_index()
    )

    x = np.arange(len(SUBSTITUTION_ORDER))
    width = 0.4

    plt.figure(figsize=(11, 5))
    plt.bar(x - width / 2, df1["weighted_frequency"], width=width, label=group_1)
    plt.bar(x + width / 2, df2["weighted_frequency"], width=width, label=group_2)
    plt.xticks(x, SUBSTITUTION_ORDER, rotation=45, ha="right")
    plt.xlabel("Substitution type")
    plt.ylabel("DB-weighted frequency")
    plt.title(f"{group_1} vs {group_2}
{group_col}")
    plt.legend()
    plt.tight_layout()

    output_path = PLOTS_DIR / f"comparison_{group_col}_{group_1}_vs_{group_2}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_spectrum_difference(spectrum_df, group_col, reference_group, comparison_group):
    ref = (
        spectrum_df[
            (spectrum_df["group_col"] == group_col)
            & (spectrum_df["group_name"] == reference_group)
        ]
        .set_index("substitution_type")
        .reindex(SUBSTITUTION_ORDER)
        .reset_index()
    )

    comp = (
        spectrum_df[
            (spectrum_df["group_col"] == group_col)
            & (spectrum_df["group_name"] == comparison_group)
        ]
        .set_index("substitution_type")
        .reindex(SUBSTITUTION_ORDER)
        .reset_index()
    )

    diff = comp["weighted_frequency"].values - ref["weighted_frequency"].values
    x = np.arange(len(SUBSTITUTION_ORDER))

    plt.figure(figsize=(10, 5))
    plt.bar(x, diff)
    plt.axhline(0, linewidth=1)
    plt.xticks(x, SUBSTITUTION_ORDER, rotation=45, ha="right")
    plt.xlabel("Substitution type")
    plt.ylabel("Difference in DB-weighted frequency")
    plt.title(f"{comparison_group} - {reference_group}
{group_col}")
    plt.tight_layout()

    output_path = PLOTS_DIR / f"difference_{group_col}_{comparison_group}_minus_{reference_group}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


## Основные графики

Ниже строятся спектры для основной классификации `spectrum_group_primary_T95`: neutral-like группа против out-of-neutral-domain группы.

In [ ]:
GROUP_COL = "spectrum_group_primary_T95"
GROUP_1 = "expanded_neutral_like_T95"
GROUP_2 = "unlabeled_out_of_neutral_domain_T95"

plot_single_spectrum(spectrum_df, GROUP_COL, GROUP_1)
plot_single_spectrum(spectrum_df, GROUP_COL, GROUP_2)
plot_two_group_spectrum(spectrum_df, GROUP_COL, GROUP_1, GROUP_2)
plot_spectrum_difference(spectrum_df, GROUP_COL, reference_group=GROUP_1, comparison_group=GROUP_2)


## Выходные файлы

Основная таблица спектров сохраняется в:

```text
../results/mutational_spectrum_results/mutation_spectrum_db_weighted_frequencies_T95_phyloP100way_v1.tsv
```

Графики сохраняются в:

```text
../results/mutational_spectrum_results/plots_db_weighted/
```